# 02 — Model Training
**Explainable Multimodal Quantum-GNN Framework for Multi-Disease Chest X-ray Diagnosis**

This notebook:
1. Sets up the environment and checks GPU
2. Mounts Drive / clones the repo
3. Loads splits (from `data/splits/`) and the HuggingFace dataset
4. Runs all 8 experiment variants in sequence
5. Compares results in a summary table
6. Plots training curves
7. Saves all results


## Cell 1 — Environment Setup

In [ ]:
# ── GPU check ────────────────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Install dependencies ─────────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    'timm',
    'datasets',
    'scikit-learn',
    'pandas',
    'matplotlib',
    'PyYAML',
    'Pillow',
    'pennylane',
    'pennylane-lightning',
    'torch-geometric',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies installed.')

## Cell 2 — Mount Drive / Clone Repo

In [ ]:
import os
import subprocess
import sys

REPO_URL  = 'https://github.com/pchandraswaroop/UROP.git'
REPO_PATH = '/content/UROP'

# Check if already inside repo root
if os.path.exists('config/config.yaml') and os.path.exists('data/splits/train.csv'):
    REPO_PATH = os.getcwd()
    print(f'Already in repository root: {REPO_PATH}')
else:
    if not os.path.exists(REPO_PATH):
        print(f'Cloning {REPO_URL} into {REPO_PATH}...')
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_PATH])
    else:
        print(f'Repo already exists at {REPO_PATH}. Pulling latest updates from GitHub...')
        try:
            subprocess.check_call(['git', '-C', REPO_PATH, 'pull', 'origin', 'main'])
        except Exception as e:
            print(f'Note on git pull: {e}')
    os.chdir(REPO_PATH)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print('Working directory:', os.getcwd())
print('Verified data/splits/train.csv exists:', os.path.exists('data/splits/train.csv'))


## Cell 3 — Load Splits and HuggingFace Dataset

In [ ]:
import os
import pandas as pd
from pathlib import Path
import yaml

# Ensure working directory is repo root if running in Colab
if not os.path.exists('config/config.yaml') and os.path.exists('/content/UROP'):
    os.chdir('/content/UROP')

# ── Config ───────────────────────────────────────────────────────────────────
with open('config/config.yaml') as f:
    CFG = yaml.safe_load(f)

SPLIT_DIR = CFG['splits']['split_dir']   # 'data/splits'
HF_REPO   = CFG['dataset']['hf_repo']   # 'chehablab/NIHChestXR'
HF_CACHE  = CFG['dataset']['cache_dir'] # 'data/raw'

os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)

train_csv = Path(SPLIT_DIR) / 'train.csv'
val_csv   = Path(SPLIT_DIR) / 'val.csv'
test_csv  = Path(SPLIT_DIR) / 'test.csv'

# ── Check & automatically build splits if not present ────────────────────────
if not (train_csv.exists() and val_csv.exists() and test_csv.exists()):
    print(f"Splits not found in '{SPLIT_DIR}'. Generating patient-level splits automatically...")
    from src.dataset.hf_loader import load_hf_dataset
    from src.dataset.splitter import build_index_dataframe, PatientSplitter, save_split_csvs

    ds = load_hf_dataset(repo=HF_REPO, cache_dir=HF_CACHE)
    index_df = build_index_dataframe(ds)
    splitter = PatientSplitter(train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=42)
    split_df = splitter.split(index_df)
    save_split_csvs(split_df, SPLIT_DIR)
    print(f"Splits successfully created and saved to '{SPLIT_DIR}'.")

# ── Training Mode Selector ───────────────────────────────────────────────────
# Set QUICK_TEST = True for fast demo training (1,500 scans, completes in ~3 mins)
# Set QUICK_TEST = False for full training across all 112,120 scans
QUICK_TEST = False
SAMPLE_SIZE = 1500

# ── Load splits ───────────────────────────────────────────────────────────────
train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
test_df  = pd.read_csv(test_csv)

if QUICK_TEST:
    print(f'⚡ QUICK TEST MODE: Streaming only first {SAMPLE_SIZE:,} scans...')
    from src.dataset.hf_loader import load_hf_dataset
    hf_dataset = load_hf_dataset(repo=HF_REPO, cache_dir=HF_CACHE, sample_n=SAMPLE_SIZE)
    train_df = train_df[train_df['hf_index'] < SAMPLE_SIZE].copy()
    val_df   = val_df[val_df['hf_index'] < SAMPLE_SIZE].copy()
    test_df  = test_df[test_df['hf_index'] < SAMPLE_SIZE].copy()
    print(f'⚡ Quick splits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
else:
    from datasets import load_dataset
    print(f'Full training mode: loading all scans from {HF_REPO}...')
    hf_dataset = load_dataset(HF_REPO, split='train', cache_dir=HF_CACHE)
    print(f'Splits: train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}')
    print(f'HF dataset: {len(hf_dataset):,} rows')


## Cell 4 — Run All Experiments

In [ ]:
import logging, json, time
from pathlib import Path
from src.training.experiment import Experiment

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%H:%M:%S')

# ── Controlled 3-Model Experiment Config ────────────────────────────────────────
BASE_CFG = {
    'batch_size':       32 if device == 'cuda' else 4,
    'num_workers':       2 if device == 'cuda' else 0,
    'learning_rate':    1e-4,
    'weight_decay':     1e-5,
    'amp':              device == 'cuda',
    'pretrained':       True,
    'quantum_enabled':  device == 'cuda',   # PennyLane quantum circuit on GPU
    'quantum_output_dim': 64,
    'clinical_output_dim': 64,
    'fused_dim':        128,
    'num_classes':      14,
    'use_weighted_loss': True,
    'pos_weight_cap':   10.0,
    'early_stopping_patience': 4,
    'save_top_k':       3,
    'checkpoint_dir':   'checkpoints',
    'log_dir':          'logs',
    'image_size':       224,
}

QUICK_TEST = False

EXPERIMENTS = [
    'cnn_only',                 # 1. Classical Vision Baseline
    'classical_multimodal',     # 2. Classical Multimodal Baseline
    'quantum_gnn_multimodal',   # 3. Proposed Quantum-GNN Architecture
]

MAX_EPOCHS = 15

print(f"Starting Controlled 3-Model Comparison Training ({len(EXPERIMENTS)} models, max {MAX_EPOCHS} epochs each):")
for exp_name in EXPERIMENTS:
    print(f" - {exp_name}")

all_results = {}   # name -> {val_metrics, test_metrics}
trained_models = {}

for exp_name in EXPERIMENTS:
    print(f"\n{'='*70}")
    print(f"  Starting Experiment: {exp_name}")
    print(f"{'='*70}")

    exp = Experiment(name=exp_name, cfg=BASE_CFG.copy(), device=device)
    exp.setup(train_df, val_df, test_df, hf_dataset)

    val_m  = exp.run(max_epochs=MAX_EPOCHS)
    test_m = exp.evaluate_test()
    exp.save_results('results')

    all_results[exp_name] = {
        'val':  val_m,
        'test': test_m,
    }
    trained_models[exp_name] = exp.model

    print(f"  Best Epoch: {getattr(exp, 'best_epoch', 0) + 1}")
    print(f"  Val  AUC={val_m.get('auc_mean', float('nan')):.4f}  F1={val_m.get('f1_macro', float('nan')):.4f}")
    print(f"  Test AUC={test_m.get('auc_mean', float('nan')):.4f}  F1={test_m.get('f1_macro', float('nan')):.4f}")

print("\nAll 3 comparative experiments complete!")



## Cell 5 — Compare Results Table

In [ ]:
from pathlib import Path
import pandas as pd
from src.training.evaluator import load_experiment_metrics, THREE_MODELS

out_dir = Path("outputs/model_comparison")
out_dir.mkdir(parents=True, exist_ok=True)

df_comparison = load_experiment_metrics(results_dir="results", models=EXPERIMENTS)
csv_path = out_dir / "comparison_metrics.csv"
df_comparison.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}\n")
print(df_comparison.to_string(index=False))
df_comparison



## Cell 6 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from src.training.evaluator import generate_comparison_plots

out_dir = Path("outputs/model_comparison")
plots = generate_comparison_plots(results_dir="results", output_dir=out_dir, models=EXPERIMENTS)

for name, path in plots.items():
    print(f"Generated: {path}")

# Display key plots inline
for img_name in ["training_curves.png", "validation_auc.png", "test_auc_comparison.png", "macro_f1_comparison.png", "disease_wise_auc.png"]:
    p = out_dir / img_name
    if p.exists():
        img = plt.imread(str(p))
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(img_name, fontsize=12, fontweight="bold")
        plt.show()



## Cell 7 — Save All Results

In [ ]:
import json
from pathlib import Path
from datetime import datetime

Path('results').mkdir(exist_ok=True)

# Full results JSON
results_path = 'results/all_results.json'
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f'Saved: {results_path}')

# Timestamp run
ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
meta = {
    'timestamp': ts,
    'device':    device,
    'experiments': EXPERIMENTS,
}
with open('results/run_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('All results saved to results/')
print(f'Run completed at {ts}')

## Cell 8 — Real Test-Set Image Predictions & Grad-CAM++ Visualizations


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from src.training.evaluator import visualize_test_predictions, generate_comparison_report

out_dir = Path("outputs/model_comparison")
ex_dir = out_dir / "prediction_examples"

print("Generating real test-set radiograph prediction comparisons with Grad-CAM++...")
saved_examples = visualize_test_predictions(
    test_df=test_df,
    hf_dataset=hf_dataset,
    models_dict=trained_models,
    output_dir=ex_dir,
    device=device,
    threshold=0.5,
    n_examples=10,
)

# Generate Markdown Comparison Report
report_path = out_dir / "comparison_report.md"
generate_comparison_report(df_comparison, report_path)
print(f"Saved comprehensive report: {report_path}")

# Display first 2 test visual examples inline
for p in saved_examples[:2]:
    img = plt.imread(str(p))
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

